In [1]:
import os
%pwd

'/workspaces/summarizer/textsummariser/research'

In [2]:
os.chdir("../")


In [3]:
%pwd

'/workspaces/summarizer/textsummariser'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [5]:
from src.textSummariser.constants import *
from src.textSummariser.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            eval_strategy = params.eval_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        return model_trainer_config



In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

In [8]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        # loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=str(self.config.root_dir),  # FIXED: Wrapped in str()
            num_train_epochs=1, 
            warmup_steps=500,
            per_device_train_batch_size=1, 
            per_device_eval_batch_size=1,
            weight_decay=0.01, 
            logging_steps=10,
            eval_strategy='steps', 
            eval_steps=500, 
            save_steps=1e6,
            gradient_accumulation_steps=16,
            fp16=False,                  
            gradient_checkpointing=True 
        ) 

        trainer = Trainer(
            model=model_pegasus, 
            args=trainer_args,
            processing_class=tokenizer, # FIXED: Updated to processing_class for Transformers v5
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"].select(range(20)),
            eval_dataset=dataset_samsum_pt["validation"].select(range(10))
        )
        
        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

In [9]:
%pip install --upgrade accelerate
%pip uninstall -y transformers accelerate
%pip install transformers accelerate

Note: you may need to restart the kernel to use updated packages.
Found existing installation: transformers 5.17.0
Uninstalling transformers-5.17.0:
  Successfully uninstalled transformers-5.17.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]2 [accelerate]
Note: you may need to restart the kernel to use updated packages.


In [10]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()

[2026-09-28 08:37:08,697: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-09-28 08:37:08,780: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-28 08:37:08,796: INFO: common: created directory at: artifacts]
[2026-09-28 08:37:08,807: INFO: common: created directory at: artifacts/model_trainer]


[2026-09-28 08:37:09,049: INFO: _client: HTTP Request: HEAD https://huggingface.co/google-t5/t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-28 08:37:09,061: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google-t5/t5-small/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/config.json?%2Fgoogle-t5%2Ft5-small%2Fresolve%2Fmain%2Fconfig.json=&etag=%22378f74060b20f6a9a1ea41b03fc0504466206255%22 "HTTP/1.1 200 OK"]
[2026-09-28 08:37:09,217: INFO: _client: HTTP Request: HEAD https://huggingface.co/google-t5/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-09-28 08:37:09,226: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-09-28 08:37:09,244: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google-t5/t5-small/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/tokenizer_config.json?%2Fgoogle-t5%2Ft5-small%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%224fd6c49c32ecf41886dba213d40030f063c72b07%22 "HTTP/1.1 200 OK"]
[2026-09-28 08:37:09,361: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-09-28 08:37:09,460: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"]
[2026-09-28 08:37:10,329: INFO: _client: HTTP Request: HEAD https://huggingface.co/google-t5/t5-small/resolve/main/confi

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/workspaces/summarizer/textsummariser/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
2,No log,2.989956


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]